# Phân loại cảm xúc review phim IMDb bằng LSTM

**Đề số 3** — Embedding + LSTM + Linear; mở rộng GRU, BiLSTM, GloVe.
**Nhóm:** Thư (leader), Bảo, Đại.

Notebook này **chỉ đọc kết quả** trong `experiments/` và `data/sample/`, không train lại. Section nào chưa có file sẽ hiện dòng ⏳ ghi rõ đang chờ ai.

| Section | Nội dung | Mục báo cáo |
| :--- | :--- | :--- |
| 1 | Vấn đề và phạm vi | 1 |
| 2 | Dữ liệu và EDA | 2 |
| 3 | Baseline | 6 |
| 4 | Mô hình | 3 |
| 5 | Theo dõi quá trình huấn luyện | 3 |
| 6 | Đánh giá, so sánh với baseline | 6 |
| 7 | Phân tích lỗi | 7 |
| 8 | Giải thích mô hình | 7 |
| 9 | Demo | 5 |
| 10 | Giới hạn | 7 |

Cách cài đặt và chạy lại toàn bộ (mục 4 của báo cáo): xem `README.md`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
EXP, RES, PLOTS = ROOT / "experiments", ROOT / "experiments" / "results", ROOT / "experiments" / "plots"

# Tên thư mục thí nghiệm theo docs/MODELS.md
MODELS = {
    "lstm": "M1 LSTM",
    "gru": "M2 GRU",
    "bilstm": "M3 BiLSTM",
    "bilstm_glove_frozen": "M4a BiLSTM + GloVe (đóng băng)",
    "bilstm_glove_ft": "M4b BiLSTM + GloVe (fine-tune)",
}
SEEDS = [42, 123, 2025]
BEST_EXP = None  # None = tự chọn theo val acc (section 6); hoặc ghi tên, ví dụ "bilstm_glove_ft"

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def rel(p: Path) -> str:
    return p.relative_to(ROOT).as_posix()


def waiting(p: Path, who: str) -> None:
    display(Markdown(f"> ⏳ Chưa có `{rel(p)}` — chờ {who}."))


def read_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))


def show_img(p: Path, who: str, width: int = 760) -> None:
    display(Image(filename=str(p), width=width)) if p.exists() else waiting(p, who)


def metrics_from_pred(df: pd.DataFrame) -> dict:
    # Metric từ file dự đoán (cột label, pred, prob theo hợp đồng ở docs/TASKS.md §4).
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    return {
        "n": len(df),
        "accuracy": accuracy_score(df["label"], df["pred"]),
        "macro_f1": f1_score(df["label"], df["pred"], average="macro"),
        "roc_auc": roc_auc_score(df["label"], df["prob"]),
    }


def pred_file(exp: str, split: str) -> Path:
    return RES / exp / f"predictions_{split}.csv"

print("ROOT =", ROOT)

## 1. Vấn đề và phạm vi

- **Bài toán:** phân loại nhị phân cảm xúc review phim tiếng Anh: tích cực (1) hoặc tiêu cực (0).
- **Dữ liệu:** IMDb Large Movie Review Dataset (Maas et al., 2011): 25.000 review train và 25.000 review test, cân bằng nhãn. Review tiêu cực có điểm ≤ 4, tích cực có điểm ≥ 7; không có review trung tính (5–6).
- **Mô hình bắt buộc:** Embedding → LSTM → Linear. **Mở rộng:** GRU, BiLSTM, BiLSTM + GloVe (đóng băng / fine-tune), thử độ dài cắt `max_len` 200/400/600.
- **Baseline:** majority class và TF-IDF (1–2 gram) + Logistic Regression.
- **Thước đo chính:** accuracy (dữ liệu cân bằng); kèm macro-F1, ROC-AUC, confusion matrix. Chọn checkpoint và early stopping theo **val accuracy**.
- **Ngoài phạm vi:** review trung tính, đa ngôn ngữ, mô hình Transformer (chỉ trích số DistilBERT từ Lab 3 để tham chiếu).

## 2. Dữ liệu và EDA

### 2.1 Chia tập và làm sạch

- Val tách từ train gốc **theo phim** (StratifiedGroupKFold 10 fold, seed 42, fold 0), để một phim không nằm ở cả train và val. Dùng lại split của Lab 3 (`data/splits/train_val_split.json`).
- Loại review trùng nội dung (SHA-1 của văn bản đã làm sạch): trùng trong train/val giữ 1 bản; trùng giữa train/val và test bỏ bản ở train/val, **giữ nguyên test 25.000**.
- Bỏ review train/val của phim cũng có trong test (`tt0406816`).

In [ ]:
st = read_json(RES / "data_prep" / "prepare_stats.json")
splits = ["train", "val", "test"]
tbl = pd.DataFrame({
    "ban đầu": st["inputs"],
    "trùng trong split": st["removed_within_split"],
    "trùng với test": st["removed_cross_split"],
    "phim trùng test": {k: v for k, v in st["removed_movie_leak"].items() if k in splits},
    "còn lại": st["outputs"],
}).reindex(splits).fillna(0).astype(int)
display(tbl)
c = st["contract"]
print(f"Tổng {c['n_rows']:,} review; tỉ lệ nhãn 1: {c['pos_share_by_split']}")
print(f"Kiểm rò rỉ: trùng hash train/val–test = {c['hash_overlap_trainval_test']}, "
      f"phim chung train–val = {c['movie_overlap_train_val']}, train–test = {c['movie_overlap_train_test']}")
print(f"Review test trùng nội dung với train gốc: {st['test_reviews_also_in_train']} (báo thêm kết quả test khi bỏ nhóm này, section 6)")

### 2.2 Thống kê tổng quan

In [ ]:
summary = pd.read_csv(RES / "eda" / "eda_summary.csv").set_index("split")
display(summary[["n_reviews", "n_pos", "n_neg", "n_movies", "len_mean", "len_median", "len_p95",
                 "len_pct_over_200", "len_pct_over_400", "pct_with_br", "pct_with_negation"]].round(1))

In [ ]:
for name in ["label_distribution", "length_hist", "length_by_label", "length_extremes"]:
    show_img(PLOTS / "eda" / f"eda_{name}.png", "Bảo (B2)")

**Nhận xét độ dài:** khoảng 41% review train dài hơn 200 từ và khoảng 13% dài hơn 400 từ. Vì vậy chọn `max_len` ảnh hưởng tới việc mô hình có đọc được câu kết luận ở cuối review hay không. Thí nghiệm `max_len` 200/400/600 ở section 6 kiểm tra điều này.

### 2.3 Phủ định, từ phổ biến, điểm số

In [ ]:
display(pd.read_csv(RES / "eda" / "eda_negation.csv"))
show_img(PLOTS / "eda" / "eda_negation_share.png", "Bảo (B2)")
show_img(PLOTS / "eda" / "eda_top_words.png", "Bảo (B2)")
display(pd.read_csv(RES / "eda" / "eda_ratings.csv").set_index("rating"))

- Phủ định xuất hiện ở khoảng 89% review, ở cả hai nhãn. Do đó không thể dựa riêng vào từ "not" để đoán nhãn. Mô hình tuần tự phải học được phủ định đổi nghĩa từ đi sau nó.
- Không có điểm 5–6, nên mô hình chưa từng thấy review trung tính (xem giới hạn, section 10).

### 2.4 Vấn đề chất lượng dữ liệu và mẫu dữ liệu

In [ ]:
display(pd.read_csv(RES / "eda" / "eda_quality_issues.csv"))
sample = pd.read_csv(ROOT / "data" / "sample" / "sample_100.csv")
print(sample.shape, "— data/sample/sample_100.csv")
display(sample.head(5))

## 3. Baseline

- **M0a Majority:** luôn đoán nhãn nhiều nhất trong train.
- **M0b TF-IDF (1–2 gram, `min_df=2`) + Logistic Regression:** chọn `C` trên val trong lưới {0,01; 0,1; 1; 10; 30; 100}.

In [ ]:
bm = read_json(RES / "baseline" / "metrics.json")
rows = {"M0a Majority": bm["M0a_majority"]["val"], f"M0b TF-IDF + LR (C={bm['M0b_tfidf_lr']['selected_C']:g})": bm["M0b_tfidf_lr"]["val"]}
display(pd.DataFrame({k: {m: v[m] for m in ["n", "accuracy", "macro_f1", "roc_auc"]} for k, v in rows.items()}).T.round(4))

grid = pd.read_csv(RES / "baseline" / "c_grid.csv")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(grid["C"], grid["train_accuracy"], "o-", label="train")
ax.plot(grid["C"], grid["val_accuracy"], "o-", label="val")
ax.set(xscale="log", xlabel="C", ylabel="accuracy", title="TF-IDF + LR: chọn C trên val")
ax.legend(); plt.show()

cm = np.array(bm["M0b_tfidf_lr"]["val"]["confusion_matrix"]["matrix"])
print("Confusion matrix M0b trên val (hàng = thật 0/1, cột = đoán 0/1):\n", cm)

Baseline TF-IDF + LR đạt **val acc ≈ 0,899**, nhưng train acc ≈ 0,999 nên overfit khoảng 10 điểm. Đây là mốc mô hình LSTM phải vượt.

## 4. Mô hình

Cùng một khung `SentimentRNN`: **Embedding → RNN → lấy trạng thái ẩn cuối → Dropout → Linear → 1 logit** (BCEWithLogitsLoss). Chỉ thay loại RNN, số chiều và embedding:

| Mã | Mô hình | Embedding | RNN |
| :--- | :--- | :--- | :--- |
| M1 | LSTM (bắt buộc) | học từ đầu, 100d | LSTM 1 chiều, 128 |
| M2 | GRU | học từ đầu, 100d | GRU 1 chiều, 128 |
| M3 | BiLSTM | học từ đầu, 100d | LSTM 2 chiều, 128 × 2 |
| M4a | BiLSTM + GloVe | GloVe 100d, đóng băng | LSTM 2 chiều, 128 × 2 |
| M4b | BiLSTM + GloVe | GloVe 100d, fine-tune | LSTM 2 chiều, 128 × 2 |

Mỗi cấu hình chạy 3 seed (42, 123, 2025), báo mean ± std. Chi tiết siêu tham số: `docs/MODELS.md`, `configs/`.

In [ ]:
rows = []
for exp, name in MODELS.items():
    p = RES / exp / "summary.json"
    if p.exists():
        rows.append(pd.json_normalize(read_json(p)).assign(model=name).set_index("model"))
    else:
        waiting(p, "Đại (D6)")
if rows:
    display(pd.concat(rows).T)

## 5. Theo dõi quá trình huấn luyện

Mỗi epoch ghi `history.csv`: `epoch, train_loss, train_acc, val_loss, val_acc, val_f1, grad_norm_mean, lr, epoch_seconds`. Với mỗi cấu hình, chồng 3 seed lên cùng hình:

- **Loss và accuracy train/val:** đánh dấu epoch tốt nhất theo val acc.
- **Khoảng cách train acc − val acc:** cho thấy mức overfit.
- **Grad norm:** phát hiện nổ gradient.

In [ ]:
def find_history(exp, seed):
    # results/ được commit; runs/ nằm trong .gitignore nên chỉ có trên máy đã train
    for p in [RES / exp / f"seed_{seed}" / "history.csv", EXP / "runs" / exp / f"seed_{seed}" / "history.csv"]:
        if p.exists():
            return pd.read_csv(p)
    return None


def plot_curves(exp, name):
    hs = {s: find_history(exp, s) for s in SEEDS}
    hs = {s: h for s, h in hs.items() if h is not None}
    if not hs:
        waiting(RES / exp / "seed_<s>" / "history.csv", "Đại (D5–D6)")
        return None
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
    rows = []
    for s, h in hs.items():
        best = h.loc[h["val_acc"].idxmax()]
        l, = axes[0].plot(h["epoch"], h["train_loss"], "--", alpha=0.7)
        axes[0].plot(h["epoch"], h["val_loss"], "-", color=l.get_color(), label=f"seed {s}")
        axes[1].plot(h["epoch"], h["train_acc"], "--", color=l.get_color(), alpha=0.7)
        axes[1].plot(h["epoch"], h["val_acc"], "-", color=l.get_color())
        axes[1].scatter([best["epoch"]], [best["val_acc"]], color=l.get_color(), marker="*", s=120, zorder=3)
        axes[2].plot(h["epoch"], h["train_acc"] - h["val_acc"], color=l.get_color())
        axes[3].plot(h["epoch"], h["grad_norm_mean"], color=l.get_color())
        rows.append({"seed": s, "best_epoch": int(best["epoch"]), "stop_epoch": int(h["epoch"].max()),
                     "best_val_acc": best["val_acc"], "val_loss@best": best["val_loss"],
                     "min_val_loss_epoch": int(h.loc[h["val_loss"].idxmin(), "epoch"]),
                     "train_minutes": h["epoch_seconds"].sum() / 60})
    for ax, t in zip(axes, ["loss (nét đứt = train)", "accuracy (★ = best val)", "train acc − val acc", "grad norm"]):
        ax.set(title=t, xlabel="epoch")
    axes[0].legend(fontsize=8)
    fig.suptitle(name); plt.tight_layout(); plt.show()
    return pd.DataFrame(rows).set_index("seed").round(4)


for exp, name in MODELS.items():
    t = plot_curves(exp, name)
    if t is not None:
        display(t)

**Cách đọc:**
- Nếu `min_val_loss_epoch` sớm hơn `best_epoch` thì val loss đã tăng trong khi val acc vẫn tăng. Mô hình tự tin quá mức trên một số mẫu, nhưng số mẫu đúng vẫn tăng. Đây là lý do nhóm chọn checkpoint theo val acc thay vì val loss.
- Khoảng cách train − val tăng dần nghĩa là overfit.
- *(Thư điền nhận xét khi có hình.)*

In [ ]:
for p in sorted((PLOTS / "training").glob("*.png")) if (PLOTS / "training").exists() else []:
    print(rel(p)); show_img(p, "Đại")

## 6. Đánh giá và so sánh với baseline

Quy trình: chọn cấu hình và checkpoint **chỉ dựa trên val**, sau đó chạy test **một lần**. Bảng so sánh và kiểm định McNemar do Đại tạo (`src/evaluate.py`). Notebook đọc lại các file đó và tự tính lại metric từ file dự đoán để đối chiếu.

In [ ]:
for split in ["val", "test"]:
    p = RES / f"comparison_{split}.csv"
    if p.exists():
        display(Markdown(f"**Bảng so sánh — {split}** (`{rel(p)}`)")); display(pd.read_csv(p))
    else:
        waiting(p, "Đại (D7–D8)")

Tham chiếu ngoài (không chạy lại): **DistilBERT fine-tune, Lab 3: test acc 0,930–0,934**. Mô hình Transformer có tiền huấn luyện được đưa vào để thấy trần hiệu năng; nó không phải đối thủ cùng hạng với LSTM.

In [ ]:
# Chọn mô hình tốt nhất theo val acc từ file dự đoán val (nếu chưa ghi tay BEST_EXP)
val_rows = {}
for exp, name in MODELS.items():
    if pred_file(exp, "val").exists():
        val_rows[exp] = metrics_from_pred(pd.read_csv(pred_file(exp, "val")))
if pred_file("baseline", "val").exists():
    val_rows["baseline"] = metrics_from_pred(pd.read_csv(pred_file("baseline", "val")))
display(pd.DataFrame(val_rows).T.round(4))

cands = {k: v["accuracy"] for k, v in val_rows.items() if k in MODELS}
if BEST_EXP is None and cands:
    BEST_EXP = max(cands, key=cands.get)
print("Mô hình tốt nhất (theo val acc):", MODELS.get(BEST_EXP, "⏳ chưa có mô hình nào"))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

if BEST_EXP and pred_file(BEST_EXP, "test").exists():
    test = pd.read_csv(pred_file(BEST_EXP, "test"))
    dup = set((RES / "data_prep" / "test_uids_dup_train.txt").read_text(encoding="utf-8").split())
    res = {"test (25.000)": metrics_from_pred(test),
           f"test bỏ {len(dup)} review trùng train": metrics_from_pred(test[~test["uid"].isin(dup)])}
    display(pd.DataFrame(res).T.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    ConfusionMatrixDisplay.from_predictions(test["label"], test["pred"], display_labels=["neg", "pos"],
                                            ax=axes[0], colorbar=False, values_format="d")
    axes[0].set_title(f"{MODELS[BEST_EXP]} — test")
    RocCurveDisplay.from_predictions(test["label"], test["prob"], ax=axes[1], name=MODELS[BEST_EXP])
    if pred_file("baseline", "test").exists():
        b = pd.read_csv(pred_file("baseline", "test"))
        RocCurveDisplay.from_predictions(b["label"], b["prob"], ax=axes[1], name="M0b TF-IDF + LR")
    plt.tight_layout(); plt.show()
else:
    waiting(RES / (BEST_EXP or "<exp>") / "predictions_test.csv", "Đại (D8)")

In [ ]:
p = RES / "mcnemar.json"
if p.exists():
    display(pd.json_normalize(read_json(p)).T)
else:
    waiting(p, "Đại (D8)")

*(Thư điền nhận xét: mô hình tốt nhất hơn baseline bao nhiêu, có ý nghĩa thống kê không (McNemar), GRU so với LSTM, BiLSTM có lợi không, GloVe đóng băng so với fine-tune, ảnh hưởng `max_len`.)*

## 7. Phân tích lỗi

Phân tích trên test chỉ để **mô tả**, không quay lại tinh chỉnh mô hình. Có 7 nhóm tình huống (`docs/phases/EVAL.md`):

1. Độ dài: < 100, 100–300, 300–600, > 600 từ (cắt ngắn có làm mất câu kết luận không).
2. Phủ định.
3. Khen chê lẫn lộn, đổi giọng ở cuối.
4. Mỉa mai, châm biếm.
5. Điểm sát ngưỡng (4, 7) so với điểm cực trị (1, 10).
6. Nhiều từ OOV hoặc từ hiếm.
7. 30 câu tự viết (`data/sample/handwritten_30.csv`).

Bảng đầy đủ và ví dụ sai kèm giải thích do `src/eval/error_analysis.py` (T2) tạo, lưu ở `experiments/results/error_analysis/`.

In [ ]:
EA = RES / "error_analysis"
files = sorted(EA.glob("*.csv")) if EA.exists() else []
for p in files:
    display(Markdown(f"**`{rel(p)}`**")); display(pd.read_csv(p).head(15))
if not files:
    waiting(EA, "Thư (T2), sau khi Đại có file dự đoán test")

### 7.1 Xem nhanh theo độ dài và điểm số

Phần này tính trực tiếp từ file dự đoán. Nếu chưa có dự đoán test của mô hình tốt nhất thì dùng dự đoán val của baseline.

In [ ]:
if BEST_EXP and pred_file(BEST_EXP, "test").exists():
    src_name, df = f"{MODELS[BEST_EXP]} — test", pd.read_csv(pred_file(BEST_EXP, "test"))
else:
    src_name, df = "M0b TF-IDF + LR — val (tạm, chờ mô hình)", pd.read_csv(pred_file("baseline", "val"))
df["correct"] = df["label"] == df["pred"]
df["n_words"] = df["text"].str.split().str.len()
df["len_bin"] = pd.cut(df["n_words"], [0, 100, 300, 600, np.inf], labels=["<100", "100–300", "300–600", ">600"])
df["rating_grp"] = df["rating"].map(lambda r: "sát ngưỡng (4, 7)" if r in (4, 7) else ("cực trị (1, 10)" if r in (1, 10) else "khác"))
print("Nguồn:", src_name)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, col in zip(axes, ["len_bin", "rating_grp", "rating"]):
    g = df.groupby(col, observed=True)["correct"].agg(["mean", "size"])
    ax.bar(g.index.astype(str), g["mean"])
    for i, (m, n) in enumerate(zip(g["mean"], g["size"])):
        ax.text(i, m, f"{m:.3f}\nn={n}", ha="center", va="bottom", fontsize=8)
    ax.set(title=f"accuracy theo {col}", ylim=(0.5, 1.05))
plt.tight_layout(); plt.show()

In [ ]:
# Lỗi tự tin nhất: mô hình sai nhưng xác suất xa 0,5
wrong = df[~df["correct"]].assign(confidence=lambda d: (d["prob"] - 0.5).abs()).nlargest(5, "confidence")
display(wrong[["uid", "label", "rating", "prob", "n_words", "text"]].assign(text=lambda d: d["text"].str[:300]))

### 7.2 Ba mươi câu tự viết

In [ ]:
hw = pd.read_csv(ROOT / "data" / "sample" / "handwritten_30.csv")
p = RES / "error_analysis" / "handwritten_30_predictions.csv"
if p.exists():
    hp = pd.read_csv(p)
    display(hp.groupby("case_type").apply(lambda g: (g["label"] == g["pred"]).mean()).rename("accuracy").to_frame())
    display(hp)
else:
    display(pd.crosstab(hw["case_type"], hw["label"]))
    waiting(p, "Thư (T2), cần checkpoint của Đại")

## 8. Giải thích mô hình

Dùng **occlusion**: che lần lượt từng từ rồi đo xác suất thay đổi bao nhiêu. Kiểm chứng bằng **deletion test**: bỏ k từ quan trọng nhất rồi xem xác suất có giảm nhanh hơn khi bỏ k từ ngẫu nhiên không. Kiểm thêm "đường tắt": mô hình có dựa vào tên diễn viên hoặc độ dài review không. Các hình do `src/eval/explain.py` (T3) tạo.

In [ ]:
pngs = sorted((PLOTS / "explain").glob("*.png")) if (PLOTS / "explain").exists() else []
for p in pngs:
    print(rel(p)); show_img(p, "Thư (T3)")
if not pngs:
    waiting(PLOTS / "explain", "Thư (T3), cần checkpoint của Đại")

## 9. Demo

Dùng hàm `predict(text) -> (label, prob)` trong `src/predict.py` (Bảo, B6). Đây cũng là hàm mà giao diện Gradio `python -m src.demo.app` gọi. Nếu chưa có hàm này thì tạm dùng baseline TF-IDF + LR, nếu trên máy có checkpoint `experiments/checkpoints/baseline_tfidf_lr.joblib`.

In [ ]:
DEMO = [
    "I expected nothing and still came away disappointed; the plot falls apart within the first twenty minutes.",
    "Not a bad film at all. The ending surprised me and I would happily watch it again.",
    "The cast is wonderful, but the script gives them nothing to do and the last act is a mess.",
] + pd.read_csv(ROOT / "data" / "sample" / "handwritten_30.csv").groupby("case_type").head(1)["text"].tolist()

try:
    from src.predict import predict
    who = "src.predict"
except ImportError:
    predict, ck = None, EXP / "checkpoints" / "baseline_tfidf_lr.joblib"
    if ck.exists():
        import joblib
        from src.data.clean import clean_text
        bl = joblib.load(ck)
        def predict(text):
            prob = float(bl["model"].predict_proba(bl["vectorizer"].transform([clean_text(text)]))[0, 1])
            return int(prob >= 0.5), prob
        who = "baseline TF-IDF + LR (tạm)"
if predict is None:
    waiting(ROOT / "src" / "predict.py", "Bảo (B6)")
else:
    print("Đang dùng:", who)
    display(pd.DataFrame([(t, *predict(t)) for t in DEMO], columns=["text", "label", "prob"]).round(3))

## 10. Giới hạn

*(Thư điền sau khi có số liệu; dưới đây là các ý đã biết trước.)*

- **Dữ liệu phân cực:** IMDb không có review điểm 5–6, nên mô hình không được kiểm trên review trung tính.
- **Cắt độ dài:** review dài hơn `max_len` bị cắt. Nếu kết luận nằm ở cuối review thì có thể bị mất (xem section 7.1).
- **Mỉa mai, khen chê lẫn lộn:** mô hình dựa nhiều vào từ cảm xúc bề mặt (xem section 7, 30 câu tự viết).
- **Ngoài miền:** chỉ huấn luyện trên review phim tiếng Anh. Kết quả trên review sản phẩm hoặc nhà hàng chỉ mang tính tham khảo.
- **Tham chiếu:** DistilBERT (Lab 3) vẫn cao hơn. LSTM nhẹ và nhanh hơn, nhưng thua về độ chính xác.